# Stacked model: stage-2 XGBoost on stage-1 probabilities

Built on the best model so far, the *context-only* variant of `xgboost_context.ipynb` (2022 micro AUC **0.9806**, 198 rounds).

**Idea.** Each row is still scored on its own, so the model cannot see (a) how profitable the *neighbouring hours* look for
the same unit, which is what decides whether a unit rides through a short dip, nor (b) what the *other units* are doing in
the same hour, although the four Tokke and three Vinje units are dispatched in order. Stage 1 already estimates both as
probabilities; stage 2 gets them as features.

**Stage-2 features** = all stage-1 features + for every unit: its stage-1 probability at hours *t−6 … t+6* (edge-padded
within the case), its rolling mean over ±3 h and ±12 h, and for Tokke and Vinje the expected number of units on at *t*.

**Out-of-fold without leakage.** Stage-1 predictions for year *y* come from a stage-1 model trained on cases before *y*
(forward chaining, with the usual one-week gap). A prediction therefore never depends on its own week's labels or on any
later year, so the honest protocol (rounds on 2021, score on 2022) stays clean. 2015 has no earlier year, so stage 2 is
trained on 2016–2022. Test predictions use a stage-1 model trained on all years.

In [ ]:
# run from the repo root so that data/, results/ and src/ resolve (notebooks live in notebooks/)
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from scipy.ndimage import uniform_filter1d
from sklearn.metrics import roc_auc_score

from kaggle_metric import score
from train_xgb_baseline import load, FeatureBuilder, H, honest_fit, split_masks, SEED
from features_margin import add_margin_features
from features_constraints import load_constraints, ConstraintFeatures, add_constraint_features
from features_context import add_context_features

REFERENCE_AUC, STAGE1_ROUNDS = 0.9806, 198     # context-only model, honest protocol
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "figure.dpi": 110})
BLUE, GREY = "#2a78d6", "#9aa3ad"

def pick_device():
    try:
        xgb.XGBClassifier(n_estimators=1, device="cuda").fit(np.zeros((4, 1)), [0, 1, 0, 1]); return "cuda"
    except Exception:
        return "cpu"
DEVICE = pick_device(); print("training device:", DEVICE)

## 1. Stage-1 features (best model so far)

In [ ]:
uc, price, inflow, volume, water_value = load("data/kernel")
minflow, minvol = load_constraints("data/extended")
test_map = pd.read_csv("prediction_mapping.csv", parse_dates=["starttime"])
sample = pd.read_csv("sample_submission.csv", nrows=1)
label_cols = [c for c in sample.columns if c != "Run No"]
units = list(dict.fromkeys(c.rsplit("_t", 1)[0].replace("result_committed_", "") for c in label_cols))
N_UNITS = len(units)
plants = [u.rsplit("_G", 1)[0] for u in units]

build_base = FeatureBuilder(price, inflow, volume, water_value)
constraints = ConstraintFeatures(minflow, minvol, volume)
def build_stage1(starts):
    return add_context_features(add_constraint_features(add_margin_features(build_base(starts)), starts, constraints))
def to_wide(pred_long, run_ids):
    wide = pred_long.reshape(len(run_ids), H, N_UNITS).transpose(0, 2, 1).reshape(len(run_ids), -1)
    out = pd.DataFrame(wide, columns=label_cols); out.insert(0, "Run No", np.asarray(run_ids)); return out

X1 = build_stage1(uc["starttime"])
Y = uc[label_cols].to_numpy(dtype=np.int8).reshape(len(uc), N_UNITS, H).transpose(0, 2, 1).reshape(-1, N_UNITS)
starts = uc["starttime"]
years_case = starts.dt.year.to_numpy()
years_row = np.repeat(years_case, H)
params = dict(learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, min_child_weight=5)
print("stage-1 features:", X1.shape)

## 2. Forward-chained out-of-fold stage-1 predictions

In [ ]:
def fit_stage1(rows, n_rounds=STAGE1_ROUNDS):
    return xgb.XGBClassifier(n_estimators=n_rounds, tree_method="hist", device=DEVICE, random_state=SEED, **params).fit(X1[rows], Y[rows])

oof = np.full((len(X1), N_UNITS), np.nan, dtype=np.float32)
stage1_auc = {}
t0 = time.time()
for y in range(2016, 2023):
    tr, va = (np.repeat(m, H) for m in split_masks(starts, y))      # train: weeks ending before Jan 1 of y; predict: year y
    m = fit_stage1(tr)
    oof[va] = m.predict_proba(X1[va])
    stage1_auc[y] = roc_auc_score(Y[va], oof[va], average="micro")
    print(f"stage-1 for {y}: trained on {tr.sum() // H} cases, micro AUC {stage1_auc[y]:.4f}  [{time.time() - t0:.0f}s]", flush=True)
assert np.isnan(oof[years_row >= 2016]).sum() == 0
np.save("results/oof_stage1.npy", oof)
print("2022 stage-1 score (should match the context-only reference):", round(stage1_auc[2022], 4))

## 3. Stage-2 features from the probabilities

In [ ]:
LAGS = range(-6, 7)
def stack_features(X_base, P):
    """P: (rows, N_UNITS) stage-1 probabilities, rows ordered (case, hour). Returns X_base + stacked columns."""
    n = len(P) // H
    P3 = P.reshape(n, H, N_UNITS)
    out = {}
    for j, u in enumerate(units):
        p = P3[:, :, j]
        for k in LAGS:
            if k == 0:
                shifted = p
            elif k < 0:
                shifted = np.concatenate([np.repeat(p[:, :1], -k, axis=1), p[:, :k]], axis=1)
            else:
                shifted = np.concatenate([p[:, k:], np.repeat(p[:, -1:], k, axis=1)], axis=1)
            out[f"s1_{u}_lag{k:+d}"] = shifted.ravel()
        out[f"s1_{u}_mean3"] = uniform_filter1d(p, size=7, axis=1, mode="nearest").ravel()      # ±3 h
        out[f"s1_{u}_mean12"] = uniform_filter1d(p, size=25, axis=1, mode="nearest").ravel()    # ±12 h
    for pl in ["Tokke", "Vinje"]:
        idx = [j for j, u in enumerate(units) if u.startswith(pl + "_")]
        out[f"s1_{pl}_units_on"] = P3[:, :, idx].sum(2).ravel()
    return pd.concat([X_base.reset_index(drop=True), pd.DataFrame(out, dtype=np.float32)], axis=1)

rows2 = years_row >= 2016
X2 = stack_features(X1[rows2], oof[rows2])
Y2, starts2 = Y[rows2], starts[years_case >= 2016].reset_index(drop=True)
stacked_cols = [c for c in X2.columns if c.startswith("s1_")]
print(f"stage-2 matrix: {X2.shape} ({len(stacked_cols)} stacked features), cases {len(starts2)}")

## 4. Stage 2 with the honest protocol (rounds on 2021, report on 2022)

In [ ]:
t0 = time.time()
res = honest_fit(X2, Y2, starts2, params, tune_year=2021, report_year=2022, device=DEVICE)
print(f"rounds chosen on 2021: {res['n_rounds']}   2021 micro AUC {res['tune_auc']:.4f}")
print(f"2022 report micro ROC AUC: {res['report_auc']:.4f}   (stage-1 reference {REFERENCE_AUC:.4f}, {res['report_auc'] - REFERENCE_AUC:+.4f})   [{time.time() - t0:.0f}s]")
val_runs = uc.loc[split_masks(starts, 2022)[1], "Run No"]
assert abs(score(to_wide(Y2[res["val_mask"]], val_runs), to_wide(res["pred"], val_runs)) - res["report_auc"]) < 1e-9

In [ ]:
stage1_2022 = oof[years_row == 2022]
per_unit = pd.DataFrame({"stage1": [roc_auc_score(Y2[res["val_mask"]][:, j], stage1_2022[:, j]) for j in range(N_UNITS)],
                         "stacked": res["per_unit"]}, index=units)
per_unit["diff"] = per_unit["stacked"] - per_unit["stage1"]
per_unit.sort_values("stacked").round(3)

In [ ]:
def transition_error_share(P, Yv):
    n = len(P) // H
    P3, Y3 = P.reshape(n, H, N_UNITS), Yv.reshape(n, H, N_UNITS)
    se = (P3 - Y3) ** 2
    near = np.zeros_like(Y3, dtype=bool); chg = np.diff(Y3, axis=1) != 0
    for k in range(-2, 3):
        lo, hi = max(0, k + 1), min(H, H + k); near[:, lo:hi, :] |= chg[:, lo - (k + 1):hi - (k + 1), :]
    return se[near].sum() / se.sum(), se.mean(), se[near].mean(), se[~near].mean()
Yv = Y2[res["val_mask"]].astype(float)
for name, P in [("stage 1", stage1_2022.astype(float)), ("stacked", res["pred"].astype(float))]:
    s, b, bn, bf = transition_error_share(P, Yv)
    print(f"{name:8s} error share within 2 h of a transition {s:.2f} | Brier all {b:.4f}, near transitions {bn:.4f}, elsewhere {bf:.4f}")

In [ ]:
imp = pd.Series(res["model"].get_booster().get_score(importance_type="gain")).sort_values()
top = imp.tail(25)
fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(top.index, top.values, color=[BLUE if c in stacked_cols else GREY for c in top.index], height=0.6)
ax.set_xlabel("average gain"); ax.set_title("Top-25 stage-2 features by gain (blue = stacked probabilities)"); ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()
print("share of gain from stacked features:", round(imp[imp.index.isin(stacked_cols)].sum() / imp.sum(), 3))

## 5. Final fit and submission

In [ ]:
t0 = time.time()
stage1_all = fit_stage1(np.ones(len(X1), dtype=bool))
X1_test = build_stage1(test_map["starttime"])
assert list(X1_test.columns) == list(X1.columns)
X2_test = stack_features(X1_test, stage1_all.predict_proba(X1_test))
assert list(X2_test.columns) == list(X2.columns) and int(X2_test.isna().sum().sum()) == 0

stage2_all = xgb.XGBClassifier(n_estimators=res["n_rounds"], tree_method="hist", device=DEVICE, random_state=SEED, **params).fit(X2, Y2)
submission = to_wide(stage2_all.predict_proba(X2_test), test_map["Run No"])
assert list(submission.columns) == list(sample.columns)
vals = submission[label_cols].to_numpy(); assert ((vals >= 0) & (vals <= 1)).all()
fname = f"submission_xgb_stacked_cv{res['report_auc']:.4f}.csv"
submission.to_csv(fname, index=False)
np.save("results/val_pred_2022_stacked.npy", res["pred"].astype(np.float32))
print(f"saved {fname} {submission.shape}  [{time.time() - t0:.0f}s]")